# Vector Store - ChromaDB Example

In [ ]:
# !pip install chromadb

## Corpus

In [1]:
# Neurogenesis - Wikipedia
document_1 = """
Neurogenesis is the process by which nervous system cells, the neurons, are produced by neural stem cells (NSCs).
This occurs in all species of animals except the porifera (sponges) and placozoans.
Types of NSCs include neuroepithelial cells (NECs), radial glial cells (RGCs), basal progenitors (BPs), 
intermediate neuronal precursors (INPs), subventricular zone astrocytes, and subgranular zone radial astrocytes, among others.
Neurogenesis is most active during embryonic development and is responsible for producing all the various types of neurons 
of the organism, but it continues throughout adult life in a variety of organisms. Once born, neurons do not divide (see mitosis), 
and many will live the lifespan of the animal, except under extraordinary and usually pathogenic circumstances.
"""

# Jet engines - Wikipedia
document_2 = """
A jet engine is a type of reaction engine, discharging a fast-moving jet of heated gas (usually air) that generates thrust by jet propulsion. 
While this broad definition may include rocket, water jet, and hybrid propulsion, the term jet engine typically refers to an internal combustion 
air-breathing jet engine such as a turbojet, turbofan, ramjet, pulse jet, or scramjet. In general, jet engines are internal combustion engines.
Air-breathing jet engines typically feature a rotating air compressor powered by a turbine, with the leftover power providing thrust through 
the propelling nozzle—this process is known as the Brayton thermodynamic cycle. Jet aircraft use such engines for long-distance travel. 
Early jet aircraft used turbojet engines that were relatively inefficient for subsonic flight. Most modern subsonic jet aircraft use more 
complex high-bypass turbofan engines. They give higher speed and greater fuel efficiency than piston and propeller aeroengines over long distances.
A few air-breathing engines made for high-speed applications (ramjets and scramjets) use the ram effect of the vehicle's speed instead of 
a mechanical compressor.
"""

# Jet engines - NASA
document_3 = """
All jet engines, which are also called gas turbines, work on the same principle. The engine sucks air in at the front with a fan. 
A compressor raises the pressure of the air. The compressor is made with many blades attached to a shaft. The blades spin at high speed and compress 
or squeeze the air. The compressed air is then sprayed with fuel and an electric spark lights the mixture. The burning gases expand and blast out 
through the nozzle, at the back of the engine. As the jets of gas shoot backward, the engine and the aircraft are thrust forward. 
As the hot air is going to the nozzle, it passes through another group of blades called the turbine. The turbine is attached to the same shaft as 
the compressor. Spinning the turbine causes the compressor to spin.
"""

# Tesla - company
document_4 = """
Tesla, Inc. is an American multinational automotive and clean energy company. Headquartered in Austin, Texas, it designs, manufactures and 
sells battery electric vehicles (BEVs), stationary battery energy storage devices from home to grid-scale, solar panels and solar shingles, 
and related products and services.
"""

# Tesla - person
document_5 = """
Nikola Tesla (10 July 1856 – 7 January 1943) was a Serbian-American engineer, futurist, and inventor. He is known for his contributions 
to the design of the modern alternating current (AC) electricity supply system.
"""

DOCUMENTS = [document_1, document_2, document_3, document_4, document_5]

## Vector Store

`ChromaDB` gives us the option to either provide our own embeddings, or let it take care of it for us when a document is added to the database. Here, we'll use the latter option and let `ChromaDB` do the work for us.

In [2]:
import hashlib

import chromadb

#### Connecting

In [3]:
CHROMADB_CLIENT = chromadb.Client()

#### Creating / opening a Collection

In [4]:
CHROMADB_COLLECTION = CHROMADB_CLIENT.get_or_create_collection(
    name="augmentation-docs-1",
    embedding_function=chromadb.utils.embedding_functions.ONNXMiniLM_L6_V2()
    
    # embedding_function=chromadb.utils.embedding_functions.OllamaEmbeddingFunction(
    #     url="http://192.168.1.240:11434/api/embeddings", 
    #     model_name="nomic-embed-text:latest"
    # )
)

Other embedding functions:

In [5]:
[name for name in dir(chromadb.utils.embedding_functions) if ("embedding" in name.lower()) and ("function" in name.lower())]

['AmazonBedrockEmbeddingFunction',
 'BasetenEmbeddingFunction',
 'Bm25EmbeddingFunction',
 'ChromaBm25EmbeddingFunction',
 'ChromaCloudQwenEmbeddingFunction',
 'ChromaCloudSpladeEmbeddingFunction',
 'ChromaLangchainEmbeddingFunction',
 'CloudflareWorkersAIEmbeddingFunction',
 'CohereEmbeddingFunction',
 'DefaultEmbeddingFunction',
 'EmbeddingFunction',
 'FastembedSparseEmbeddingFunction',
 'GoogleGeminiEmbeddingFunction',
 'GoogleGenaiEmbeddingFunction',
 'GoogleGenerativeAiEmbeddingFunction',
 'GooglePalmEmbeddingFunction',
 'GoogleVertexEmbeddingFunction',
 'HuggingFaceEmbeddingFunction',
 'HuggingFaceSparseEmbeddingFunction',
 'InstructorEmbeddingFunction',
 'JinaEmbeddingFunction',
 'MistralEmbeddingFunction',
 'MorphEmbeddingFunction',
 'NomicEmbeddingFunction',
 'OllamaEmbeddingFunction',
 'OpenAIEmbeddingFunction',
 'OpenCLIPEmbeddingFunction',
 'PerplexityEmbeddingFunction',
 'RoboflowEmbeddingFunction',
 'SentenceTransformerEmbeddingFunction',
 'SparseEmbeddingFunction',
 'Tex

#### Adding / replacing Documents

Documents can be added one at a time or, preferably, in batch.

In [6]:
for document in DOCUMENTS:
    
    CHROMADB_COLLECTION.add(
            ids=[hashlib.md5(document.encode()).hexdigest()[-8:]],  # Unique identifier
            documents=[document],  # Text for embedding
            metadatas=[{
                "word-count": len(document.split()),
                "source": "wikipedia"
            }],
            # embeddings=[...]     # if you want to bring "your own" embeddings
    )

In [7]:
CHROMADB_COLLECTION.count()

5

Updating is done by Id using the `update` method. Arguments are the same as `add`.

# Simple Fetching (not a search)

In [8]:
fetched_documents = CHROMADB_COLLECTION.get(ids=None, include=["documents", "metadatas", "embeddings"])

In [9]:
fetched_documents.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [10]:
fetched_documents["ids"]

['b747e7ec', '0b1863c7', '028b3ec2', 'd20d7366', '76b605f7']

In [11]:
fetched_documents["documents"][0]

'\nNeurogenesis is the process by which nervous system cells, the neurons, are produced by neural stem cells (NSCs).\nThis occurs in all species of animals except the porifera (sponges) and placozoans.\nTypes of NSCs include neuroepithelial cells (NECs), radial glial cells (RGCs), basal progenitors (BPs), \nintermediate neuronal precursors (INPs), subventricular zone astrocytes, and subgranular zone radial astrocytes, among others.\nNeurogenesis is most active during embryonic development and is responsible for producing all the various types of neurons \nof the organism, but it continues throughout adult life in a variety of organisms. Once born, neurons do not divide (see mitosis), \nand many will live the lifespan of the animal, except under extraordinary and usually pathogenic circumstances.\n'

In [12]:
fetched_documents["metadatas"][0]

{'source': 'wikipedia', 'word-count': 115}

In [13]:
fetched_documents["embeddings"][0]

array([-7.64335915e-02, -1.29148111e-01, -3.27299722e-02, -2.35357676e-02,
       -3.24141607e-02,  3.80648561e-02, -1.17644602e-02, -1.08012250e-02,
        4.38548252e-02,  1.09020285e-01,  3.74334306e-02, -5.75723276e-02,
       -8.98112133e-02,  3.63836773e-02, -1.11842811e-01,  6.29647970e-02,
       -4.89951000e-02, -4.90292981e-02,  3.35700298e-03,  4.52114753e-02,
        2.45154221e-02,  6.69975253e-03, -2.55923755e-02, -2.85852719e-02,
       -1.75790545e-02,  3.81759880e-03,  5.07344666e-04, -1.77615285e-02,
        8.02705139e-02, -7.08802487e-04,  5.52446060e-02,  1.14319455e-02,
       -1.91958081e-02,  7.21799303e-03,  1.49203846e-02,  1.63769796e-02,
       -1.30024347e-02, -5.43443114e-02,  2.71950141e-02,  2.86088753e-02,
        2.60671135e-02, -2.50649638e-02, -2.77956333e-02,  6.90292194e-02,
        5.82872666e-02,  9.97599121e-03,  3.94021161e-02, -6.59244508e-02,
       -3.27739939e-02, -1.01850823e-01, -1.08319046e-02, -7.90721476e-02,
       -3.47960703e-02,  

<hr/>